In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append('/content/drive/MyDrive/CS3268_Project/src')
import numpy as np, pandas as pd, xgboost as xgb
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score
from common import (load_split, features, train_model, threshold_at_fpr, PREDS, ROOT, CATEGORICALS, LABEL)
np.random.seed(42)
RUN_FIX = False   # gate: set True only if the core (CIs, SHAP, own Method/Results) is done (Fri 6 Nov night)

In [ ]:
# 1. Load. Variant II categoricals are cast to the BASE category sets so frozen models see the same codes.
base = {s: load_split(s) for s in ("train", "val", "test")}
v2 = {s: load_split(s, "variant2") for s in ("train", "val", "test")}
cats = {c: base["train"][c].cat.categories for c in CATEGORICALS}
def align(df):
    df = df.copy()
    for c in CATEGORICALS:
        df[c] = df[c].astype(pd.CategoricalDtype(cats[c]))
    return df
v2 = {s: align(d) for s, d in v2.items()}
print({c: int(v2["val"][c].isna().sum()) for c in CATEGORICALS})   # expect all 0 (unseen categories become NaN)

# Frozen models, loaded from Drive (no retraining in the check)
def load(path):
    m = xgb.XGBClassifier(enable_categorical=True); m.load_model(path); return m
frozen = {"baseline": load(f"{ROOT}/models/baseline.json"), "reweigh": load(f"{ROOT}/models/reweigh.json")}
ew = pd.read_csv(f"{ROOT}/models/expgrad_weights.csv")
eg_comps = [(r.weight, load(f"{ROOT}/models/expgrad_components/comp_{int(r.component)}.json")) for r in ew.itertuples()]
def score(name, X):
    if name == "expgrad":
        return sum(w * m.predict_proba(X)[:, 1] for w, m in eg_comps) / ew.weight.sum()
    return frozen[name if name != "groupthr" else "baseline"].predict_proba(X)[:, 1]

In [ ]:
# 2. DETECT the shift (no training). Base val (month 5) vs Variant II month 5: same month, no time confound.
def tv(a, b):
    p, q = a.value_counts(normalize=True), b.value_counts(normalize=True)
    return 0.5 * p.sub(q, fill_value=0).abs().sum()
def shift_table(A, B):
    XA, XB = features(A), features(B)
    d = {c: (tv(XA[c], XB[c]) if c in CATEGORICALS else ks_2samp(XA[c].dropna(), XB[c].dropna()).statistic) for c in XA.columns}
    return pd.Series(d).sort_values(ascending=False)   # KS D (numeric) or total-variation distance (categorical); ranked, no p-value cutoff
print("Covariate shift, Base val vs Variant II month 5 (top 10):"); print(shift_table(base["val"], v2["val"]).head(10).round(3))
print("Time shift on Base, val vs test (top 10):"); print(shift_table(base["val"], base["test"]).head(10).round(3))
for nm, d in [("Base val", base["val"]), ("Variant II m5", v2["val"]), ("Base test", base["test"])]:
    print(nm, "fraud rate overall / under-50 / 50-plus:", round(d[LABEL].mean(), 4), d.groupby("age_group")[LABEL].mean().round(4).tolist())  # label shift
for nm, d in [("Base val", base["val"]), ("Variant II m5", v2["val"])]:   # concept drift hint, not a test
    s = score("baseline", features(d))
    print(nm, "baseline AUC:", round(roc_auc_score(d[LABEL], s), 4))
s_b, s_v = score("baseline", features(base["val"])), score("baseline", features(v2["val"]))
print("KS on honest baseline scores, Base val vs Variant II m5:", round(ks_2samp(s_b[(base["val"][LABEL] == 0).values], s_v[(v2["val"][LABEL] == 0).values]).statistic, 4))

In [ ]:
# 3. CHECK: frozen models, BASE cutoffs unchanged. clean = Base val; shifted = Base test and Variant II months 6-7.
def metrics(y, g, dec):
    y, g, dec = map(np.asarray, (y, g, dec))
    f = [dec[(y == 0) & (g == k)].mean() for k in (0, 1)]
    return dict(recall=dec[y == 1].mean(), fpr=dec[y == 0].mean(), fpr_ratio=min(f) / max(f),
                recall_u50=dec[(y == 1) & (g == 0)].mean(), recall_50p=dec[(y == 1) & (g == 1)].mean())
cuts = {}    # cutoffs read back from the saved Base val predictions (decision == 1 boundary)
for nm in ("baseline", "reweigh", "expgrad", "groupthr"):
    p = pd.read_parquet(f"{PREDS}/preds_{nm}.parquet"); v = p[p.month == 5]
    cuts[nm] = ({k: threshold_at_fpr(v[v.age_group == k].score, v[v.age_group == k].y_true) for k in (0, 1)}
                if nm == "groupthr" else threshold_at_fpr(v.score, v.y_true))
def decide(nm, s, g):
    c = cuts[nm]
    return (s >= np.vectorize(c.get)(g)).astype(int) if isinstance(c, dict) else (s >= c).astype(int)
v2t = v2["test"]   # Variant II months 6-7
rows = []
for nm in ("baseline", "reweigh", "groupthr", "expgrad"):
    p = pd.read_parquet(f"{PREDS}/preds_{nm}.parquet")
    clean = metrics(p[p.month == 5].y_true, p[p.month == 5].age_group, p[p.month == 5].decision)
    bt = p[p.month >= 6]
    sets = {"Base test (time)": metrics(bt.y_true, bt.age_group, bt.decision),
            "Variant II m6-7 (dataset+time)": metrics(v2t[LABEL], v2t.age_group, decide(nm, score(nm, features(v2t)), v2t.age_group.values))}
    for sh, m in sets.items():
        for k in clean: rows.append(dict(model=nm, shifted=sh, metric=k, clean=clean[k], shifted_val=m[k], R=m[k] / clean[k]))
check = pd.DataFrame(rows).round(4); print(check.to_string())   # paste this to Claude

In [ ]:
# 4. FIX (lecture: retrain on combined data). Baseline + reweigh only; ExpGrad not retrained (too slow; stated limitation).
# Train = Base train + Variant II train, frozen BEST_PARAMS, seed 42, no dataset-indicator feature.
# Cutoff(s) set at 5% FPR on Variant II val (month 5); evaluated on Variant II months 6-7 (same rows as the check).
if RUN_FIX:
    trc = pd.concat([base["train"], v2["train"]], ignore_index=True)
    trc = align(trc); g, y = trc["age_group"], trc[LABEL]
    pg, py, pgy = g.value_counts(normalize=True), y.value_counts(normalize=True), pd.crosstab(g, y, normalize=True)
    w = np.array([pg[a] * py[b] / pgy.loc[a, b] for a, b in zip(g, y)])   # reweighing weights recomputed on combined data
    Xc = features(trc)
    new = {"baseline": train_model(Xc, y), "reweigh": train_model(Xc, y, sample_weight=w)}
    rows = []
    for nm, m in new.items():
        s_val, s_te = m.predict_proba(features(v2["val"]))[:, 1], m.predict_proba(features(v2t))[:, 1]
        c = threshold_at_fpr(s_val, v2["val"][LABEL])
        out = {nm + "_retrained": (s_te >= c).astype(int)}
        if nm == "baseline":   # group thresholds re-derived from the retrained baseline scores
            gc = {k: threshold_at_fpr(s_val[(v2["val"].age_group == k).values], v2["val"][LABEL][(v2["val"].age_group == k).values]) for k in (0, 1)}
            out["groupthr_retrained"] = (s_te >= v2t.age_group.map(gc).values).astype(int)
        for lab, dec in out.items():
            m_ = metrics(v2t[LABEL], v2t.age_group, dec)
            rows.append(dict(model=lab, **{k: round(v, 4) for k, v in m_.items()}))
    before = check[check.shifted.str.startswith("Variant II")].pivot(index="model", columns="metric", values="shifted_val").round(4)
    print("BEFORE (frozen Base models, Base cutoffs), Variant II m6-7:"); print(before)
    print("AFTER (retrained on Base+Variant II, cutoff on Variant II val), Variant II m6-7:"); print(pd.DataFrame(rows))  # paste both